#  Biomass Estimation with ResNet50

**Task:** Predict dry biomass (grams) from crop images using a CNN conditioned on the target type.
**Model:** ResNet50 (pretrained on ImageNet) with a conditioned regression head.


### Table of Contents
1. Imports & Setup
2. Data Loading
3. Train / Validation Split
4. Target Scaling (StandardScaler only)
5. Image Transforms
6. Dataset Class
7. Model Architecture
8. Weighted R² Metric
9. Training & Evaluation Helpers
10. Random Search (Hyperparameter Tuning)
11. Grid Search (Refined Tuning)
12. Final Training with Best Parameters
13. Final Model Performance
14. Saving the Model
15. Plots & Interpretation


## 1. Imports & Setup

We import all required libraries and mount Google Drive to access the data and save the model.


We set **random seeds** everywhere (`random`, `numpy`, `torch`, `cuda`) to ensure experiments are **reproducible**, same seed = same splits and weight initialisations every run.

`DEVICE` auto-detects GPU (`cuda`) if available, otherwise falls back to CPU.


In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from itertools import product

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seeds(SEED)
print(f"Using device: {DEVICE}")

MessageError: Error: credential propagation was unsuccessful

## 2. Configuration & Data Loading

We define file paths following the group template structure and load the CSV files.

The 5 **target variable names** we predict:


The CSV is in **long format**: one row per image × target type, so each image appears 5 times. The model receives the image AND a one-hot vector for which target to predict.


In [ ]:
base_dir      = "/content/drive/My Drive/DL_Data/split_blanche/"
traincsv_path = "/content/drive/MyDrive/DL_Data /split_blanche /train_split.csv"
testcsv_path  = "/content/drive/MyDrive/DL_Data /split_blanche /test_split.csv"
train_pics    = "/content/drive/MyDrive/DL_Data /split_blanche /train_images"
test_pics     = "/content/drive/MyDrive/DL_Data /split_blanche /test_images"

TARGET_NAMES = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]

df = pd.read_csv(traincsv_path)
print("Total train rows:", len(df))
print("Unique train images:", df["image_path"].nunique())
df.head()

## 3. Train / Validation Split


Each image appears **5 times** in the CSV (once per target type). Splitting randomly by row risks putting the same image in both train and val, the model would have "seen" that image during training, making validation accuracy **artificially optimistic** (data leakage).

**what's the best approach then?** split on unique image paths first, then all 5 rows of an image go to the same set.

```
train_df: 80% of unique images  →  all 5 targets for those images
val_df:   20% of unique images  →  all 5 targets for those images
```

> The test set comes from `test_split.csv` and is kept completely separate,  only used for final evaluation, never during training or hyperparameter search.


In [ ]:
def split_by_image_path(df, val_size=0.2):
    unique_images = df["image_path"].unique()
    train_imgs, val_imgs = train_test_split(
        unique_images, test_size=val_size, random_state=SEED
    )
    train_df = df[df["image_path"].isin(train_imgs)].reset_index(drop=True)
    val_df   = df[df["image_path"].isin(val_imgs)].reset_index(drop=True)
    return train_df, val_df

train_df, val_df = split_by_image_path(df)
print(f"Train rows : {len(train_df)}  |  Unique images: {train_df['image_path'].nunique()}")
print(f"Val rows   : {len(val_df)}    |  Unique images: {val_df['image_path'].nunique()}")

## 4. Target Scaling: StandardScaler

Biomass values in grams are **right-skewed**, most are small but a few are very large. We apply `StandardScaler` to the raw target values:

```
z = (x - mean) / std
```

This centers targets around 0 with standard deviation 1, which:
- Helps gradient descent converge faster and more stably
- Prevents the loss from being dominated by a few very large values



In [ ]:
target_scaler = StandardScaler()

train_targets = train_df["target"].values.reshape(-1, 1)
val_targets   = val_df["target"].values.reshape(-1, 1)

target_scaler.fit(train_targets)

train_df = train_df.copy()
val_df   = val_df.copy()
train_df["target_scaled"] = target_scaler.transform(train_targets).flatten()
val_df["target_scaled"]   = target_scaler.transform(val_targets).flatten()

print(f"Scaler learned:  mean = {target_scaler.mean_[0]:.4f},  std = {target_scaler.scale_[0]:.4f}")
print(f"Train target range — original : [{train_df['target'].min():.2f}, {train_df['target'].max():.2f}] g")
print(f"Train target range — scaled   : [{train_df['target_scaled'].min():.2f}, {train_df['target_scaled'].max():.2f}]")
print(f"Val   target range — scaled   : [{val_df['target_scaled'].min():.2f},  {val_df['target_scaled'].max():.2f}]")

## 5. Image Transforms

We apply exactly two transforms:

1. **`ToTensor()`** — converts a PIL image (H × W × 3, values 0–255) to a float tensor (3 × H × W, values 0.0–1.0)
2. **`Normalize(mean, std)`** — subtracts per-channel means and divides by per-channel standard deviations



In [ ]:
# Values computed from the training set in the group preprocessing notebook
train_mean = [0.43918201, 0.49828457, 0.30579902]
train_std  = [0.24945421, 0.24771695, 0.23592069]

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std)
])

print("Transform pipeline:")
print("  1. ToTensor()     — PIL [H,W,3] uint8  →  tensor [3,H,W] float32 in [0,1]")
print("  2. Normalize()    — subtract channel mean, divide by channel std")
print(f"     mean: {train_mean}")
print(f"     std : {train_std}")

## 6. Dataset Class — `BiomassDataset`

PyTorch requires a custom `Dataset` class implementing:
- `__len__`: total number of samples
- `__getitem__`: return one sample by index

Our dataset extends the group template's `CustomImageDataset` by adding:
- **One-hot encoding** of the target type (so the model knows which of the 5 targets to predict)
- **Scaled target** column (pre-computed in cell 4)

### What does each sample contain?

| Key | Shape | Description |
|---|---|---|
| `image` | `[3, H, W]` | Normalised image tensor |
| `target_oh` | `[5]` | One-hot vector identifying the target type |
| `target` | scalar | StandardScaler-scaled biomass value |
| `target_name` | string | e.g. `"Dry_Total_g"` |

### Why a one-hot vector?
The same image is associated with 5 different targets. By concatenating this vector with the image features inside the model, a single model learns all 5 targets simultaneously — no need for 5 separate models.

```
Dry_Green_g  → [1, 0, 0, 0, 0]
Dry_Dead_g   → [0, 1, 0, 0, 0]
Dry_Clover_g → [0, 0, 1, 0, 0]
GDM_g        → [0, 0, 0, 1, 0]
Dry_Total_g  → [0, 0, 0, 0, 1]
```


In [ ]:
class BiomassDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

        # Pre-compute one-hot matrix: shape [N, 5]
        target_to_idx  = {name: i for i, name in enumerate(TARGET_NAMES)}
        target_idx     = self.df["target_name"].map(target_to_idx).values
        self.target_oh = np.eye(len(TARGET_NAMES))[target_idx].astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load image — strip subfolder prefix to get the bare filename
        img_name = row["image_path"].split("/")[-1]
        img_path = os.path.join(self.img_dir, img_name)
        image    = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return {
            "image":       image,
            "target_oh":   torch.tensor(self.target_oh[idx], dtype=torch.float32),
            "target":      torch.tensor(row["target_scaled"], dtype=torch.float32),
            "target_name": row["target_name"]
        }

# Sanity check
_ds    = BiomassDataset(train_df, train_pics, image_transform)
sample = _ds[0]
print("Image shape      :", sample["image"].shape)
print("Image dtype      :", sample["image"].dtype)
print(f"Pixel range      : [{sample['image'].min():.3f}, {sample['image'].max():.3f}]")
print("One-hot          :", sample["target_oh"].tolist())
print(f"Target (scaled)  : {sample['target'].item():.4f}")
print("Target name      :", sample["target_name"])
del _ds

## 7. Model Architecture — `ResNet50Conditioned`

### Why ResNet50?
ResNet50 is a CNN pretrained on ImageNet (1.2M images, 1000 classes). Its convolutional layers already detect edges, textures, and colour patterns. **Transfer learning** lets us fine-tune these features for biomass estimation instead of training from scratch.

### Architecture overview

```
Input image tensor [3 × H × W]
        ↓
   ResNet50 backbone (pretrained ImageNet weights)
   [49 conv/pooling layers + global average pooling]
        ↓
   Image feature vector  [2048-dim]
        ↓
   Concatenate with one-hot target type [5-dim]
        ↓
   Combined vector [2053-dim]
        ↓
   Fully Connected Regression Head:
     Linear(2053 → hidden_dim)       tunable, e.g. 512
     ReLU + Dropout(p)
     Linear(hidden_dim → hidden_dim/2)
     ReLU + Dropout(p)
     Linear(hidden_dim/2 → 1)
        ↓
   Single predicted value (scaled biomass)
```

### Key design choices
- **`backbone.fc = nn.Identity()`** — removes ResNet's 1000-class softmax head, exposing the 2048-dim feature vector
- **Conditioning on target type** — concatenating the one-hot vector tells the model which of the 5 biomass fractions to predict
- **`hidden_dim` and `dropout`** are tuned via hyperparameter search


In [ ]:
class ResNet50Conditioned(nn.Module):
    def __init__(self, dropout=0.3, hidden_dim=256):
        super().__init__()

        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_features = backbone.fc.in_features   # 2048
        backbone.fc = nn.Identity()             # remove classification head
        self.backbone = backbone

        # Regression head: 2048 image features + 5 one-hot = 2053 → 1
        self.fc = nn.Sequential(
            nn.Linear(in_features + len(TARGET_NAMES), hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, image, target_oh):
        img_feat = self.backbone(image)                # [B, 2048]
        x        = torch.cat([img_feat, target_oh], dim=1)  # [B, 2053]
        return self.fc(x).squeeze(1)                   # [B]

# Parameter count
_m        = ResNet50Conditioned()
total     = sum(p.numel() for p in _m.parameters())
trainable = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")
del _m

## 8. Weighted R² Metric

This is the **official group evaluation metric**, taken directly from the template.

### What is R²?
```
R² = 1 - SS_residuals / SS_total
   = 1 - Σ(y - ŷ)² / Σ(y - ȳ)²
```
- **R² = 1.0** → perfect predictions
- **R² = 0.0** → model just predicts the mean, no better than a naive baseline
- **R² < 0** → model is worse than predicting the mean

### Why weighted?
Different targets have different agronomic importance:

| Target | Weight | Reason |
|---|---|---|
| `Dry_Total_g` | **0.4** | Most important — total biomass |
| `GDM_g` | 0.2 | Green dry matter |
| `Dry_Green_g` | 0.2 | Living fraction |
| `Dry_Dead_g` | 0.1 | |
| `Dry_Clover_g` | 0.1 | |

### Important: this function works on raw gram values
The predictions from the model are in **scaled space** (StandardScaler output). Before calling `weighted_r2`, we must inverse-transform back to grams:
```python
preds_grams = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
```


In [ ]:
# Taken from the group template (model_template.ipynb)
TARGET_WEIGHTS = {
    'Dry_Total_g':  0.4,
    'GDM_g':        0.2,
    'Dry_Green_g':  0.2,
    'Dry_Dead_g':   0.1,
    'Dry_Clover_g': 0.1,
}

def weighted_r2(all_preds: np.ndarray, all_targets: np.ndarray, all_names: np.ndarray) -> float:
    """
    Computes the weighted R² score across all target types.
    IMPORTANT: all_preds and all_targets must be in original gram values (not scaled).
    """
    df_tmp = pd.DataFrame({
        'pred':        all_preds,
        'target':      all_targets,
        'target_name': all_names
    })

    SSres_total = 0.0
    SStot_total = 0.0
    for target_name, weight in TARGET_WEIGHTS.items():
        subset = df_tmp[df_tmp['target_name'] == target_name]
        if len(subset) == 0:
            continue
        SSres_total += weight * ((subset['target'] - subset['pred']) ** 2).sum()
        SStot_total += weight * ((subset['target'] - subset['target'].mean()) ** 2).sum()

    weighted_score = 1 - SSres_total / SStot_total
    print(f"  {'Weighted R²':20s}     {weighted_score:.4f}")
    return weighted_score

print("weighted_r2 function defined (from group template).")

## 9. Training & Evaluation Helpers

### `train_one_epoch`
One full pass through the training data:
1. Load batch → move to GPU
2. Forward pass → predictions (in scaled space)
3. Compute MSE loss between scaled predictions and scaled targets
4. Backpropagate → update weights with AdamW
5. Return average loss over the epoch

### `evaluate_full`
Runs inference on a data loader (`@torch.no_grad()` — no gradients needed):
- Collects all scaled predictions, scaled targets, and target names
- **Inverse-transforms** both back to grams before calling `weighted_r2`
- Returns MSE (scaled space) and weighted R² (gram space)

### `run_trial`
Full training run for a given set of hyperparameters — used by both random search and grid search.

Unlike a quick proxy, this version:
- Trains for however many epochs are specified in `params["epochs"]`
- Applies **early stopping** via `params["patience"]`: stops if val MSE hasn't improved for that many consecutive epochs
- Saves the best model weights (`state_dict`) seen during training
- Returns a **result dict** with `params`, `best_val_mse`, `best_val_r2`, `best_epoch`, and `state_dict`

This richer return value is what allows both search functions to build a proper summary table and save the best model.


In [ ]:
import itertools

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        images    = batch["image"].to(DEVICE)
        target_oh = batch["target_oh"].to(DEVICE)
        y         = batch["target"].to(DEVICE)

        optimizer.zero_grad()
        preds = model(images, target_oh)
        loss  = criterion(preds, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_full(model, loader, scaler):
    """
    Evaluates the model on a data loader.
    - MSE is computed in scaled space (fast, used for monitoring)
    - weighted_r2 is computed in original gram space (the real metric)
    """
    model.eval()
    preds_scaled, targets_scaled, names = [], [], []
    total_loss = 0
    criterion  = nn.MSELoss()

    for batch in loader:
        images    = batch["image"].to(DEVICE)
        target_oh = batch["target_oh"].to(DEVICE)
        y         = batch["target"].to(DEVICE)
        outputs   = model(images, target_oh)
        loss      = criterion(outputs, y)
        total_loss += loss.item() * images.size(0)
        preds_scaled.extend(outputs.cpu().numpy())
        targets_scaled.extend(y.cpu().numpy())
        names.extend(batch["target_name"])

    mse = total_loss / len(loader.dataset)

    # Inverse-transform to grams before computing weighted_r2
    preds_grams   = scaler.inverse_transform(np.array(preds_scaled).reshape(-1, 1)).flatten()
    targets_grams = scaler.inverse_transform(np.array(targets_scaled).reshape(-1, 1)).flatten()

    r2 = weighted_r2(preds_grams, targets_grams, np.array(names))
    return mse, r2


def run_trial(params):
    """
    Trains a model with the given hyperparameter dict and returns a result dict.
    Used by both random_search() and grid_search().

    params must contain: seed, lr, weight_decay, epochs, patience, batch_size,
                         hidden_dim, dropout
    Returns: {params, best_val_mse, best_val_r2, best_epoch, state_dict}
    """
    set_seeds(params["seed"])

    train_ds = BiomassDataset(train_df, train_pics, image_transform)
    val_ds   = BiomassDataset(val_df,   train_pics, image_transform)
    t_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True,
                          num_workers=2, pin_memory=True)
    v_loader = DataLoader(val_ds,   batch_size=params["batch_size"], shuffle=False,
                          num_workers=2, pin_memory=True)

    model = ResNet50Conditioned(
        dropout    = params["dropout"],
        hidden_dim = params["hidden_dim"]
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr           = params["lr"],
        weight_decay = params["weight_decay"]
    )
    criterion = nn.MSELoss()

    best_val_mse     = float("inf")
    best_val_r2      = -999
    best_epoch       = -1
    best_state       = None
    patience_counter = 0

    for epoch in range(params["epochs"]):
        train_one_epoch(model, t_loader, optimizer, criterion)
        val_mse, val_r2 = evaluate_full(model, v_loader, target_scaler)

        if val_mse < best_val_mse:
            best_val_mse     = val_mse
            best_val_r2      = val_r2
            best_epoch       = epoch + 1
            best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= params["patience"]:
            print(f"    Early stopping at epoch {epoch + 1}")
            break

    del model
    torch.cuda.empty_cache()

    return {
        "params":       params,
        "best_val_mse": best_val_mse,
        "best_val_r2":  best_val_r2,
        "best_epoch":   best_epoch,
        "state_dict":   best_state,
    }

print("Training helpers defined.")


## 10. Random Search — Hyperparameter Tuning (Phase 1)




In [ ]:
# Random search explores a random subset of all possible combinations.
# More efficient than full grid search when the search space is large.

def random_search(n_trials=10):
    search_space = {
        "seed":         [42],
        "lr":           [3e-5, 1e-4, 3e-4],
        "weight_decay": [1e-5, 1e-4],
        "epochs":       [10, 15, 20],
        "patience":     [3],
        "batch_size":   [32, 64],
        "hidden_dim":   [128, 256],
        "dropout":      [0.1, 0.2, 0.3],
    }

    keys       = list(search_space.keys())
    all_combos = [dict(zip(keys, combo))
                  for combo in itertools.product(*search_space.values())]
    total_combos = len(all_combos)
    print(f"Total possible random-search combinations: {total_combos}")

    if n_trials > total_combos:
        print(f"Requested n_trials={n_trials}, but only {total_combos} unique combinations exist.")
        n_trials = total_combos

    # Sample without replacement — avoids testing the same combination twice
    sampled_combos = random.sample(all_combos, n_trials)

    results = []
    best    = None

    for trial, params in enumerate(sampled_combos, start=1):
        print(f"\n===== Random trial {trial}/{n_trials} =====")
        print(params)
        result = run_trial(params)
        results.append(result)

        if (best is None) or (result["best_val_mse"] < best["best_val_mse"]):
            best = result
            print("New best val MSE:", best["best_val_mse"], "| val R²:", best["best_val_r2"])

    summary = pd.DataFrame([
        {
            **r["params"],
            "best_val_mse": r["best_val_mse"],
            "best_val_r2":  r["best_val_r2"],
            "best_epoch":   r["best_epoch"],
        }
        for r in results
    ]).sort_values("best_val_mse", ascending=True).reset_index(drop=True)

    return best, summary, results


# ----------------------------
# Run random search
# ----------------------------
best_random, random_summary, random_results = random_search(n_trials=10)

print("\nTop random search results:")
print(random_summary.head(10))
random_summary.to_csv("random_search_summary.csv", index=False)


## 11. Grid Search — Hyperparameter Tuning (Phase 2)

### Strategy: exhaustive search over a curated, hand-selected grid

Unlike the random search which samples from a large space, the grid search **tries every combination** in a smaller, deliberately chosen grid. The values here are informed by prior knowledge and the random search results, but the grid is defined independently — it does not automatically use the random search output.

| Hyperparameter | Grid values | Why these values |
|---|---|---|
| `seed` | 42 | Fixed for reproducibility |
| `lr` | 1e-4, 3e-4 | Range around what worked in random search |
| `weight_decay` | 1e-4, 1e-2 | Light vs stronger regularisation |
| `epochs` | 15, 20 | Enough to converge with early stopping |
| `patience` | 5 | More tolerance than random search (longer training) |
| `batch_size` | 32 | Fixed — larger batches didn't help |
| `hidden_dim` | 256, 512 | Medium vs larger regression head |
| `dropout` | 0.1, 0.2 | Low dropout (backbone already regularised) |

Total combinations: 2 × 2 × 2 × 1 × 1 × 2 × 2 = **32**

### How best model selection works
- Best within grid search = lowest `best_val_mse` across all combos
- After both searches, we compare `best_random` vs `best_grid` by val MSE and take the overall winner
- The winning `state_dict` is what gets saved to disk

### What to look for in the output
- Combos with `lr=3e-4` + `hidden_dim=512` tend to perform best (consistent with your random search findings)
- `weight_decay=1e-2` provides stronger regularisation and can improve generalisation
- A summary table is saved to `grid_search_summary.csv`


In [ ]:
# Grid search tries every combination in a smaller, self-selected search grid.
# More computationally costly than random search, but exhaustive within the grid.

def grid_search():
    grid = {
        "seed":         [42],
        "lr":           [1e-4, 3e-4],
        "weight_decay": [1e-4, 1e-2],
        "epochs":       [15, 20],
        "patience":     [5],
        "batch_size":   [32],
        "hidden_dim":   [256, 512],
        "dropout":      [0.1, 0.2],
    }

    keys   = list(grid.keys())
    combos = list(itertools.product(*grid.values()))
    print(f"Total grid combinations: {len(combos)}")

    results = []
    best    = None

    for i, combo in enumerate(combos, start=1):
        params = dict(zip(keys, combo))
        print(f"\n===== Grid trial {i}/{len(combos)} =====")
        print(params)
        result = run_trial(params)
        results.append(result)

        if (best is None) or (result["best_val_mse"] < best["best_val_mse"]):
            best = result
            print("New best val MSE:", best["best_val_mse"], "| val R²:", best["best_val_r2"])

    summary = pd.DataFrame([
        {
            **r["params"],
            "best_val_mse": r["best_val_mse"],
            "best_val_r2":  r["best_val_r2"],
            "best_epoch":   r["best_epoch"],
        }
        for r in results
    ]).sort_values("best_val_mse", ascending=True).reset_index(drop=True)

    return best, summary, results


# ----------------------------
# Run grid search
# ----------------------------
best_grid, grid_summary, grid_results = grid_search()

print("\nTop grid search results:")
print(grid_summary.head(10))
grid_summary.to_csv("grid_search_summary.csv", index=False)


## 12. Selecting the Best Overall Model

After both searches, we compare the best result from random search and the best result from grid search — both measured by **val MSE** — and take the winner.

This is the model that gets trained further and saved. The source (random or grid) is recorded so results are fully reproducible.


In [ ]:
# ----------------------------
# Pick best overall by val MSE
# ----------------------------
if best_grid["best_val_mse"] <= best_random["best_val_mse"]:
    best_overall = best_grid
    best_source  = "grid_search"
else:
    best_overall = best_random
    best_source  = "random_search"

print(f"Best overall source  : {best_source}")
print(f"Best val MSE         : {best_overall['best_val_mse']:.4f}")
print(f"Best val R²          : {best_overall['best_val_r2']:.4f}")
print(f"Best params          : {best_overall['params']}")


## 13. Final Training with Best Parameters

We take the best hyperparameters found by the searches and do a **full final training run** for 20 epochs — this time tracking full history for plotting.

The model checkpoint is saved whenever a new best val R² is achieved.




In [ ]:
best_params = best_overall["params"]

print("\n" + "="*60)
print(f"FINAL TRAINING — best params from {best_source}:")
print(best_params)
print("="*60)

set_seeds(best_params["seed"])
BATCH_SIZE = best_params["batch_size"]
EPOCHS     = 20

train_ds_final = BiomassDataset(train_df, train_pics, image_transform)
val_ds_final   = BiomassDataset(val_df,   train_pics, image_transform)
train_loader   = DataLoader(train_ds_final, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=2, pin_memory=True)
val_loader     = DataLoader(val_ds_final,   batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True)

model = ResNet50Conditioned(
    dropout    = best_params["dropout"],
    hidden_dim = best_params["hidden_dim"]
).to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = best_params["lr"],
    weight_decay = best_params["weight_decay"]
)
criterion = nn.MSELoss()

history    = {"train_loss": [], "val_loss": [], "val_r2": []}
best_r2    = -999
best_state = None
best_epoch = -1

for epoch in range(EPOCHS):
    train_loss       = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_r2 = evaluate_full(model, val_loader, target_scaler)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_r2"].append(val_r2)

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | R2: {val_r2:.4f}")

    if val_r2 > best_r2:
        best_r2    = val_r2
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch + 1

model.load_state_dict(best_state)
print(f"\nBest checkpoint: epoch {best_epoch}  |  Val R² = {best_r2:.4f}")


## 13. Final Model Performance

We run `evaluate_full` one last time on the **best model** (weights from epoch 10) to get the definitive validation metrics.




In [ ]:
print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE (validation set)")
print("="*60)
final_mse, final_r2 = evaluate_full(model, val_loader, target_scaler)
print(f"  MSE (scaled space) : {final_mse:.6f}")
print("="*60)
print(f"\nThe model explains {final_r2*100:.1f}% of biomass variance")
print(f"   (weighted R², evaluated on raw gram values, across all 5 targets)")

## 14. Saving the Full Model

### Why save the full model object and not just the weights?

| Approach | How to load | Use case |
|---|---|---|
| `state_dict` only | Must re-define the class first | Fine if you have the codebase |
| **Full model object** | `model = checkpoint['model']` | No class definition needed — ideal for sharing |

The checkpoint contains everything needed to reproduce results or run inference:

| Key | Content |
|---|---|
| `model` | Full PyTorch model object |
| `state_dict` | Weights backup |
| `best_params` | Hyperparameters used |
| `best_epoch` | Epoch that achieved the best R² |
| `best_r2` | Best validation R² |
| `target_scaler` | **Essential for inference** — needed to inverse-transform scaled predictions back to grams |
| `target_names` | The 5 target type names |

### How to load and run inference

```python
import torch, numpy as np
checkpoint = torch.load('best_resnet50_conditioned_full.pth', map_location='cpu')
model  = checkpoint['model'].eval()
scaler = checkpoint['target_scaler']

with torch.no_grad():
    pred_scaled = model(image_tensor.unsqueeze(0), target_oh.unsqueeze(0))
    pred_grams  = scaler.inverse_transform(pred_scaled.cpu().numpy().reshape(-1, 1)).flatten()
```


In [ ]:
save_path = "/content/drive/My Drive/DL_Data/best_resnet50_conditioned_full.pth"

torch.save({
    "model":         model,
    "state_dict":    best_state,
    "best_params":   best_params,
    "best_epoch":    best_epoch,
    "best_r2":       best_r2,
    "target_scaler": target_scaler,
    "target_names":  TARGET_NAMES,
}, save_path)

print(f"Full model saved to:\n   {save_path}")
print("\nTo load:")
print("  checkpoint = torch.load('best_resnet50_conditioned_full.pth')")
print("  model  = checkpoint['model'].eval()")
print("  scaler = checkpoint['target_scaler']")

## 15. Plots & Interpretation

### Plot 1 — Training vs Validation Loss (MSE in scaled space)

**What to look for:**
- Both curves decreasing together → healthy learning
- Large gap between train and val by end → overfitting
- Red dashed line = best epoch (saved checkpoint)


In [ ]:
epochs_range = range(1, EPOCHS + 1)

# ── Plot 1: Training vs Validation Loss ───────────────────────────────────
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, history["train_loss"], label="Train Loss",      marker="o")
plt.plot(epochs_range, history["val_loss"],   label="Validation Loss", marker="o")
plt.axvline(best_epoch, color="red", linestyle="--", alpha=0.6, label=f"Best epoch ({best_epoch})")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (scaled space)")
plt.title("Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig("/content/drive/My Drive/DL_Data/loss_curve.png", dpi=150)
plt.show()
print("↑ Train loss → ~0 while val loss plateaus at ~0.19 after epoch 10 → overfitting")

# ── Plot 2: Weighted R² & MSE (dual axis) ─────────────────────────────────
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Weighted R²", color="steelblue")
ax1.plot(epochs_range, history["val_r2"],   color="steelblue",  marker="o", label="Val R²")
ax1.tick_params(axis="y", labelcolor="steelblue")
ax1.axvline(best_epoch, color="red", linestyle="--", alpha=0.6, label=f"Best epoch ({best_epoch})")

ax2 = ax1.twinx()
ax2.set_ylabel("Val MSE (scaled)", color="darkorange")
ax2.plot(epochs_range, history["val_loss"], color="darkorange", marker="s", linestyle="--", label="Val MSE")
ax2.tick_params(axis="y", labelcolor="darkorange")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right")

plt.title("Validation Weighted R² & MSE over Epochs")
fig.tight_layout()
plt.savefig("/content/drive/My Drive/DL_Data/r2_mse_curve.png", dpi=150)
plt.show()
print("↑ R² peaks at epoch 10 (0.51) then oscillates — constant LR + overfitting")
print(f"\nBest model: Epoch {best_epoch} | Weighted R² = {best_r2:.4f} | MSE = {final_mse:.4f}")

### Evaluation

In [ ]:
# ----------------------------
# Load test set
# ----------------------------
test_df = pd.read_csv(testcsv_path).copy()

# Scale targets using the scaler already fit earlier in THIS notebook
# (train-only fit, then transform test)
test_targets = test_df["target"].values.reshape(-1, 1)
if "target_scaled" not in test_df.columns:
    test_df["target_scaled"] = target_scaler.transform(test_targets).flatten()

# ----------------------------
# Test dataset / loader
# ----------------------------
test_ds = BiomassDataset(test_df, test_pics, image_transform)

cpu_count = os.cpu_count() or 0
num_workers = min(4, cpu_count)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    pin_memory=True,
    num_workers=num_workers,
    persistent_workers=(num_workers > 0),
)

# ----------------------------
# Evaluation helper
# ----------------------------
@torch.no_grad()
def evaluate_test(model, loader, scaler):
    model.eval()
    criterion = nn.MSELoss()

    preds_scaled = []
    targets_scaled = []
    target_names = []
    total_loss = 0.0

    for batch in loader:
        images = batch["image"].to(DEVICE)
        target_oh = batch["target_oh"].to(DEVICE)
        y = batch["target"].to(DEVICE)

        outputs = model(images, target_oh)
        loss = criterion(outputs, y)
        total_loss += loss.item() * images.size(0)

        preds_scaled.extend(outputs.detach().cpu().numpy())
        targets_scaled.extend(y.detach().cpu().numpy())
        target_names.extend(batch["target_name"])

    test_mse = total_loss / len(loader.dataset)

    preds_orig = scaler.inverse_transform(np.array(preds_scaled).reshape(-1, 1)).flatten()
    targets_orig = scaler.inverse_transform(np.array(targets_scaled).reshape(-1, 1)).flatten()

    global_r2 = r2_score(targets_orig, preds_orig)
    weighted_r2_score = weighted_r2(preds_orig, targets_orig, np.array(target_names))

    return test_mse, global_r2, weighted_r2_score, preds_orig, targets_orig, target_names

# ----------------------------
# Pick model from THIS notebook
# ----------------------------
if "final_model" in globals():
    eval_model = final_model
elif "model" in globals():
    eval_model = model
elif "best_state" in globals() and "best_params" in globals():
    eval_model = ResNet50Conditioned(
        dropout=best_params["dropout"],
        hidden_dim=best_params["hidden_dim"]
    ).to(DEVICE)
    eval_model.load_state_dict(best_state)
else:
    raise ValueError(
        "No trained model found in this notebook. Expected one of: final_model, model, or best_state + best_params."
    )

eval_model = eval_model.to(DEVICE)
eval_model.eval()

# ----------------------------
# Run evaluation
# ----------------------------
test_mse, global_r2, weighted_r2_score, preds_orig, targets_orig, target_names = evaluate_test(
    model=eval_model,
    loader=test_loader,
    scaler=target_scaler,
)

print("\n================ TEST RESULTS ================")
print("TEST MSE (scaled space):", test_mse)
print("TEST R² (global)       :", global_r2)
print("TEST Weighted R²       :", weighted_r2_score)

# ----------------------------
# Results dataframe
# ----------------------------
results_df = pd.DataFrame({
    "pred": preds_orig,
    "target": targets_orig,
    "name": target_names
})

# ----------------------------
# R² per target
# ----------------------------
r2_per_label = {}

for name in TARGET_NAMES:
    sub = results_df[results_df["name"] == name]
    if len(sub) > 1:
        denom = ((sub["target"] - sub["target"].mean()) ** 2).sum()
        if denom != 0:
            r2_per_label[name] = 1 - (((sub["target"] - sub["pred"]) ** 2).sum() / denom)
        else:
            r2_per_label[name] = np.nan
    else:
        r2_per_label[name] = np.nan

# ----------------------------
# Residual analysis
# ----------------------------
results_df["residual"] = results_df["target"] - results_df["pred"]
results_df["abs_error"] = np.abs(results_df["residual"])

# ----------------------------
# Plots
# ----------------------------
plt.figure(figsize=(8, 4))
plt.bar(r2_per_label.keys(), r2_per_label.values())
plt.title("R² per target")
plt.xticks(rotation=45)
plt.grid()
plt.show()

plt.figure(figsize=(5, 4))
scores = [global_r2, weighted_r2_score]
labels = ["Global R²", "Weighted R²"]
plt.bar(labels, scores)
plt.title("R² comparison")
plt.axhline(0, linestyle="--")
plt.grid()
plt.ylim(min(-1, min(scores) - 0.1), max(1, max(scores) + 0.1))
plt.show()

for name in TARGET_NAMES:
    sub = results_df[results_df["name"] == name]
    plt.figure()
    plt.hist(sub["residual"], bins=30)
    plt.title(f"Residuals: {name}")
    plt.grid()
    plt.show()

for name in TARGET_NAMES:
    sub = results_df[results_df["name"] == name]
    plt.figure()
    plt.scatter(sub["target"], sub["abs_error"], alpha=0.5)
    plt.title(f"Error vs target: {name}")
    plt.xlabel("True value")
    plt.ylabel("Absolute error")
    plt.grid()
    plt.show()

# ----------------------------
# Save results
# ----------------------------
results_df.to_csv("test_predictions.csv", index=False)
print("\nSaved: test_predictions.csv")

from google.colab import files
files.download("test_predictions.csv")

